# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mahadumar/flyrank-MLinternship-Assignment-1/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [4]:
import os
from google.colab import userdata
import duckdb
import pandas as pd
import numpy as np

# Load token from Colab secrets — never paste tokens in cells
HF_TOKEN = userdata.get("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN
print(f"Token loaded: {HF_TOKEN[:8]}...")

# Connect DuckDB to HuggingFace
con = duckdb.connect()
con.execute(f"CREATE SECRET hf_secret (TYPE huggingface, TOKEN '{HF_TOKEN}')")

# Base path — only fact_content_daily_performance is accessible via parquet
REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance/**/*.parquet"

# Development month — mid-panel, safe to use for feature/label development
# 2026-06 is the sealed test month — we never query it during development
DEV_MONTH = "2026-03"
SEALED_MONTH = "2026-06"  # defined here as a reminder never to touch it

print(f"\nDevelopment month : {DEV_MONTH}")
print(f"Sealed test month : {SEALED_MONTH}  ← never used for development")
print("\nSetup complete. Ready.")

Token loaded: hf_ePFOh...

Development month : 2026-03
Sealed test month : 2026-06  ← never used for development

Setup complete. Ready.


## 1. Unit of analysis + time window

**One row = one content page on one report date for one client.**

More precisely: the grain of `fact_content_daily_performance` is
`(report_date, client_hash_id, content_hash_id)` — a unique combination
of a single day, a single client, and a single piece of content.

For Lane 2 (Refresh / Content Opportunity Scoring), the decision unit is
a **page**, not a day. So when building features, we aggregate daily rows
into a per-page monthly summary:

- **Observation window:** `month = 2026-03` (one mid-panel month)
- **Aggregation:** sum impressions, sum clicks, average position per page
- **Unit after aggregation:** one row = one page in March 2026

**Time window reasoning:**
- We use `month = 2026-03` for all development, feature engineering, and
  label construction. This is a mid-panel month with full history on both
  sides — enough prior months to compute trend signals, enough future
  months to eventually validate.
- `month = 2026-06` is the sealed test month. It is never queried during
  development. Touching it would contaminate our honest evaluation.

**Availability filter:**
We restrict to rows where `gsc_data_available IS TRUE`. Pages without
GSC data have no impressions or position signal — they cannot be scored
for refresh opportunity. Rows where GSC is unavailable are excluded from
all feature and label construction.

In [5]:
# Verify the grain: confirm (report_date, client_hash_id, content_hash_id)
# is unique in the development month

print("GRAIN VERIFICATION — month =", DEV_MONTH)
print("=" * 60)

grain_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT
            report_date || '|' || client_hash_id || '|' || content_hash_id
        ) AS unique_combinations
    FROM read_parquet('{FACT}', hive_partitioning=true)
    WHERE month = '{DEV_MONTH}'
      AND gsc_data_available IS TRUE
""").df()

print(grain_check.to_string(index=False))

total = grain_check["total_rows"].iloc[0]
unique = grain_check["unique_combinations"].iloc[0]

if total == unique:
    print(f"\n✓ Grain confirmed: every row is unique on (date, client, content)")
else:
    print(f"\n⚠ Duplicates found: {total - unique:,} extra rows")

print(f"\nRows in dev month with GSC data: {total:,}")

GRAIN VERIFICATION — month = 2026-03


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 total_rows  unique_combinations
    3611061              3611061

✓ Grain confirmed: every row is unique on (date, client, content)

Rows in dev month with GSC data: 3,611,061


## 2. Fields: feature / label / context / excluded

### Features (inputs to the model — all knowable at decision time)

| Field | Role | Knowable at decision time because… |
|---|---|---|
| `gsc_impressions` (monthly sum) | Feature | GSC reports past impressions — fully observable before any refresh decision |
| `gsc_clicks` (monthly sum) | Feature | Same as impressions — trailing search performance, observable in arrears |
| `gsc_avg_position` (monthly mean) | Feature | Average search rank over the month — knowable before decision |
| `ga4_engaged_sessions` (monthly sum) | Feature | GA4 engagement trailing signal — observable before decision |
| `ctr` (clicks / impressions, computed) | Feature | Derived from clicks and impressions — both knowable, ratio is knowable |

### Label / Proxy

| Field | Role | Note |
|---|---|---|
| `is_declining` (computed) | Proxy label | Defined as: impressions in current month < impressions in prior month. Reasonable proxy for refresh need. Imperfect — seasonal patterns can cause decline unrelated to content quality. |

### Context (identifiers — not features, not label)

| Field | Role |
|---|---|
| `client_hash_id` | Client identifier — used for holdout split, never as a feature |
| `content_hash_id` | Page identifier — the unit of analysis |
| `report_date` | Date — used for aggregation, not a feature |
| `month` | Partition key — used for filtering, not a feature |

### Excluded (with reason)

| Field | Excluded because… |
|---|---|
| `gsc_sum_position` | Raw sum used to compute avg_position — redundant once avg is computed |
| `sessions_ai`, `ai_chatgpt`, etc. | AI referral traffic is extremely sparse in this dataset — using it as a feature would add noise, not signal |
| `ga4_data_available = FALSE` rows | No engagement signal available — excluded via filter |
| `gsc_data_available = FALSE` rows | No search signal available — cannot score refresh opportunity without it |
| `month = 2026-06` | Sealed test month — excluded from all development to prevent data contamination |

In [6]:
# Show the field availability in the dev month
# Confirm GSC and GA4 availability rates

print("FIELD AVAILABILITY — month =", DEV_MONTH)
print("=" * 60)

availability = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END)
            AS gsc_available_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END)
            AS ga4_available_rows,
        SUM(CASE WHEN gsc_data_available IS TRUE
                  AND ga4_data_available IS TRUE THEN 1 ELSE 0 END)
            AS both_available_rows
    FROM read_parquet('{FACT}', hive_partitioning=true)
    WHERE month = '{DEV_MONTH}'
""").df()

print(availability.to_string(index=False))

total = availability["total_rows"].iloc[0]
gsc = availability["gsc_available_rows"].iloc[0]
ga4 = availability["ga4_available_rows"].iloc[0]
both = availability["both_available_rows"].iloc[0]

print(f"\nGSC available:  {gsc:,} / {total:,}  ({gsc/total*100:.1f}%)")
print(f"GA4 available:  {ga4:,} / {total:,}  ({ga4/total*100:.1f}%)")
print(f"Both available: {both:,} / {total:,}  ({both/total*100:.1f}%)")
print(f"\nRows excluded (no GSC): {total-gsc:,} — these cannot be scored")

FIELD AVAILABILITY — month = 2026-03


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 total_rows  gsc_available_rows  ga4_available_rows  both_available_rows
    9841378           3611061.0            413966.0             364347.0

GSC available:  3,611,061.0 / 9,841,378  (36.7%)
GA4 available:  413,966.0 / 9,841,378  (4.2%)
Both available: 364,347.0 / 9,841,378  (3.7%)

Rows excluded (no GSC): 6,230,317.0 — these cannot be scored


## 3. Verify it with queries

Three verification queries on `month = 2026-03`, filtered with
`gsc_data_available IS TRUE` as required by the assignment.

**Query 1 — Grain check:** Confirm one row really is one page-date-client

**Query 2 — Row count and date span:** How many scoreable pages exist
in the dev month, and what date range do they cover?

**Query 3 — Availability filter with IS TRUE:** Show exactly how many
rows survive the `gsc_data_available IS TRUE` filter, and what the
key signal columns look like on the surviving rows.

In [7]:
# QUERY 1 — Grain verification
# Confirm the grain is (report_date, client_hash_id, content_hash_id)

print("QUERY 1 — Grain check")
print("=" * 60)

q1 = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT content_hash_id) AS unique_pages,
        COUNT(DISTINCT client_hash_id) AS unique_clients,
        COUNT(DISTINCT report_date) AS unique_dates
    FROM read_parquet('{FACT}', hive_partitioning=true)
    WHERE month = '{DEV_MONTH}'
      AND gsc_data_available IS TRUE
""").df()

print(q1.to_string(index=False))
print()
print("Interpretation:")
print(f"  total_rows = unique_pages × unique_dates (approximately)")
print(f"  → Grain is confirmed as (date, client, content)")

QUERY 1 — Grain check
 total_rows  unique_pages  unique_clients  unique_dates
    3611061        176738              47            31

Interpretation:
  total_rows = unique_pages × unique_dates (approximately)
  → Grain is confirmed as (date, client, content)


In [11]:
# QUERY 2 — Row count and date span in dev month

print("QUERY 2 — Row count and date span")
print("=" * 60)

q2 = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT content_hash_id) AS unique_pages,
        MIN(report_date) AS earliest_date,
        MAX(report_date) AS latest_date,
        COUNT(DISTINCT report_date) AS days_in_window,
        AVG(gsc_impressions) AS avg_daily_impressions,
        AVG(gsc_avg_position) AS avg_position
    FROM read_parquet('{FACT}', hive_partitioning=true)
    WHERE month = '{DEV_MONTH}'
      AND gsc_data_available IS TRUE
""").df()

print(q2.to_string(index=False))
print()
print(f"Dev month confirmed: {DEV_MONTH}")
print(f"These are the pages available for feature construction and labeling.")

QUERY 2 — Row count and date span
 total_rows  unique_pages earliest_date latest_date  days_in_window  avg_daily_impressions  avg_position
    3611061        176738    2026-03-01  2026-03-31              31              77.721642     15.826651

Dev month confirmed: 2026-03
These are the pages available for feature construction and labeling.


In [12]:
# QUERY 3 — Availability filter with IS TRUE
# Show exactly how the filter works and what survives

print("QUERY 3 — Availability filter (gsc_data_available IS TRUE)")
print("=" * 60)

q3 = con.sql(f"""
    SELECT
        gsc_data_available,
        ga4_data_available,
        COUNT(*) AS row_count,
        COUNT(DISTINCT content_hash_id) AS unique_pages,
        AVG(gsc_impressions) AS avg_impressions
    FROM read_parquet('{FACT}', hive_partitioning=true)
    WHERE month = '{DEV_MONTH}'
    GROUP BY gsc_data_available, ga4_data_available
    ORDER BY gsc_data_available DESC, ga4_data_available DESC
""").df()

print(q3.to_string(index=False))
print()
print("Rows where gsc_data_available IS TRUE are the only scoreable rows.")
print("All feature construction and labeling uses this filter exclusively.")

QUERY 3 — Availability filter (gsc_data_available IS TRUE)
 gsc_data_available  ga4_data_available  row_count  unique_pages  avg_impressions
               True                True     364347         63856       234.218259
               True               False    1718348        127319        48.114814
               True                <NA>    1528366         99502        73.701505
              False                True      49619         32582         0.000000
              False               False    4690323        210634         0.000000
              False                <NA>    1490375        116548         0.000000

Rows where gsc_data_available IS TRUE are the only scoreable rows.
All feature construction and labeling uses this filter exclusively.


## 4. Five features + the leakage trap

### Five features for Lane 2

Built by aggregating daily rows to monthly per-page summaries.
Each feature is justified with an "available when?" statement.

| Feature | Definition | Available when? |
|---|---|---|
| `monthly_impressions` | SUM(gsc_impressions) over the month | Knowable at decision time — GSC reports past impressions with ~2 day lag |
| `monthly_clicks` | SUM(gsc_clicks) over the month | Same as impressions — trailing observable signal |
| `avg_position` | AVG(gsc_avg_position) over the month | Average rank over the month — knowable before any refresh decision |
| `ctr` | monthly_clicks / monthly_impressions | Derived ratio — both inputs are knowable, ratio is knowable |
| `monthly_engaged_sessions` | SUM(ga4_engaged_sessions) over the month | GA4 engagement signal — trailing metric, observable before decision |

### The proxy label

`is_declining` = 1 if `monthly_impressions` in 2026-03
`monthly_impressions` in 2026-02 for the same page. Otherwise 0.

This is a reasonable proxy for "page needs refresh attention" —
a page losing impressions month-over-month is a refresh candidate.

### The trap (deliberate leakage experiment)

We will add `impressions_next_month` (March impressions from April data)
as a feature, watch Precision@20 jump toward perfect, then remove it.
This is the leakage lesson from w02 performed on real warehouse data.

In [13]:
# Build the five-feature frame for dev month
# Aggregating daily rows to monthly per-page summaries

print("Building feature frame for", DEV_MONTH)
print("=" * 60)

# Current month (observation window)
current = con.sql(f"""
    SELECT
        content_hash_id,
        client_hash_id,
        SUM(gsc_impressions)         AS monthly_impressions,
        SUM(gsc_clicks)              AS monthly_clicks,
        AVG(gsc_avg_position)        AS avg_position,
        SUM(ga4_engaged_sessions)    AS monthly_engaged_sessions
    FROM read_parquet('{FACT}', hive_partitioning=true)
    WHERE month = '{DEV_MONTH}'
      AND gsc_data_available IS TRUE
    GROUP BY content_hash_id, client_hash_id
""").df()

# Prior month (for label construction)
prior_month = "2026-02"
prior = con.sql(f"""
    SELECT
        content_hash_id,
        SUM(gsc_impressions) AS prior_impressions
    FROM read_parquet('{FACT}', hive_partitioning=true)
    WHERE month = '{prior_month}'
      AND gsc_data_available IS TRUE
    GROUP BY content_hash_id
""").df()

# Merge and build features
feat = current.merge(prior, on="content_hash_id", how="inner")

# Compute CTR (safe division)
feat["ctr"] = feat["monthly_clicks"] / feat["monthly_impressions"].replace(0, np.nan)
feat["ctr"] = feat["ctr"].fillna(0)

# Build proxy label: declining = impressions dropped month-over-month
feat["is_declining"] = (
    feat["monthly_impressions"] < feat["prior_impressions"]
).astype(int)

feature_cols = [
    "content_hash_id", "client_hash_id",
    "monthly_impressions", "monthly_clicks",
    "avg_position", "ctr",
    "monthly_engaged_sessions",
    "prior_impressions", "is_declining"
]

print(f"Feature frame shape: {feat.shape[0]:,} pages × {feat.shape[1]} columns")
print(f"\nLabel distribution:")
print(f"  is_declining = 1: {feat['is_declining'].sum():,} pages "
      f"({feat['is_declining'].mean()*100:.1f}%)")
print(f"  is_declining = 0: {(1-feat['is_declining']).sum():,} pages "
      f"({(1-feat['is_declining']).mean()*100:.1f}%)")
print(f"\nSample rows:")
print(feat[feature_cols].head(8).to_string(index=False))

Building feature frame for 2026-03


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame shape: 134,238 pages × 9 columns

Label distribution:
  is_declining = 1: 39,397 pages (29.3%)
  is_declining = 0: 94,841 pages (70.7%)

Sample rows:
         content_hash_id          client_hash_id  monthly_impressions  monthly_clicks  avg_position      ctr  monthly_engaged_sessions  prior_impressions  is_declining
content_05597932fe4da067 client_73cda7b4e4f265ea                 57.0             0.0      2.714744 0.000000                       0.0              207.0             1
content_7a105f548d9c6916 client_73cda7b4e4f265ea               6523.0             7.0      7.209549 0.001073                       0.0             4270.0             0
content_905aa32a0230694e client_73cda7b4e4f265ea                149.0             0.0      6.481453 0.000000                       0.0              156.0             1
content_a3ea9792f793ec72 client_73cda7b4e4f265ea                453.0             0.0      2.987198 0.000000                       0.0              440.0           

In [17]:
# THE LEAKAGE TRAP — deliberate experiment
# Using a simple model to prevent in-sample overfitting from masking the lesson

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import numpy as np

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

y = feat["is_declining"].values

# Scale features for logistic regression
scaler = StandardScaler()

# ── HONEST MODEL: five safe features only ─────────────────────────────────────
safe_features = [
    "monthly_impressions", "monthly_clicks",
    "avg_position", "ctr", "monthly_engaged_sessions"
]
X_safe = scaler.fit_transform(feat[safe_features].fillna(0).values)

clf_safe = LogisticRegression(
    class_weight="balanced", random_state=42, max_iter=1000
)
clf_safe.fit(X_safe, y)
scores_safe = clf_safe.predict_proba(X_safe)[:, 1]

p20_safe = precision_at_k(scores_safe, y, 20)
p50_safe = precision_at_k(scores_safe, y, 50)

print("HONEST MODEL (5 safe features, logistic regression):")
print(f"  Precision@20 = {p20_safe:.3f}")
print(f"  Precision@50 = {p50_safe:.3f}")

# ── LEAKY MODEL: add prior_impressions as a "feature" ─────────────────────────
# prior_impressions IS used to construct the label (is_declining)
# is_declining = 1 when monthly_impressions < prior_impressions
# So prior_impressions directly encodes the label boundary — pure leakage

print()
print("LEAKY MODEL (adding prior_impressions as a feature):")
print("WARNING: prior_impressions is used to CONSTRUCT the label.")
print("         is_declining = 1 when monthly_impressions < prior_impressions")
print("         The model sees the answer directly.\n")

leaky_features = safe_features + ["prior_impressions"]
X_leaky = scaler.fit_transform(feat[leaky_features].fillna(0).values)

clf_leaky = LogisticRegression(
    class_weight="balanced", random_state=42, max_iter=1000
)
clf_leaky.fit(X_leaky, y)
scores_leaky = clf_leaky.predict_proba(X_leaky)[:, 1]

p20_leaky = precision_at_k(scores_leaky, y, 20)
p50_leaky = precision_at_k(scores_leaky, y, 50)

print(f"  Precision@20 = {p20_leaky:.3f}  ← inflated by leakage")
print(f"  Precision@50 = {p50_leaky:.3f}  ← inflated by leakage")

print()
print("CONCLUSION:")
print(f"  Honest Precision@50:  {p50_safe:.3f}")
print(f"  Leaky  Precision@50:  {p50_leaky:.3f}")
print(f"  Inflation from leak:  +{p50_leaky - p50_safe:.3f}")
print()
print("prior_impressions is now REMOVED from all future modeling.")
print("The honest number is what gets reported. The leaky number is discarded.")

HONEST MODEL (5 safe features, logistic regression):
  Precision@20 = 0.350
  Precision@50 = 0.560

LEAKY MODEL (adding prior_impressions as a feature):
         is_declining = 1 when monthly_impressions < prior_impressions
         The model sees the answer directly.

  Precision@20 = 1.000  ← inflated by leakage
  Precision@50 = 1.000  ← inflated by leakage

CONCLUSION:
  Honest Precision@50:  0.560
  Leaky  Precision@50:  1.000
  Inflation from leak:  +0.440

prior_impressions is now REMOVED from all future modeling.
The honest number is what gets reported. The leaky number is discarded.


## 4. Data limits

**One named limitation of this slice:**

The panel is unbalanced — different clients have different history depths.
Some clients only joined in early 2026, meaning their pages have only 1-2
months of history. When we compute month-over-month impression change to
build the `is_declining` label, pages with no prior month data are excluded
from the inner join. This means the feature frame systematically excludes
newer clients and newer content — the model will be trained and evaluated
on pages with at least 2 months of GSC history, which is not representative
of all pages in production.

**Additional limits:**

- `gsc_data_available IS TRUE` filters out pages with no search console
  access. These pages are invisible to the model entirely — if a client
  has no GSC, none of their pages can be scored.

- The proxy label (`is_declining`) is based on a single month-over-month
  comparison. A page that declined in February but recovered in March
  would be labeled 0 (not declining) even though it may still need review.
  Seasonal patterns, crawl anomalies, and one-off traffic spikes all
  introduce noise into this label that the model cannot distinguish from
  genuine content decay.

- This notebook uses in-sample evaluation (train and score on the same
  month). Real validation requires a client-holdout split — pages from
  a given client appear in either training or evaluation, never both.
  That is the next step.

In [15]:
# Quantify the unbalanced panel limitation
# Show how many pages are EXCLUDED because they have no prior month data

print("DATA LIMIT — Unbalanced panel (missing prior month)")
print("=" * 60)

# Pages in dev month with GSC data
n_current = len(current)

# Pages that appear in both current AND prior month (our feature frame)
n_joined = len(feat)

# Pages dropped by inner join (no prior month data)
n_dropped = n_current - n_joined

print(f"Pages in {DEV_MONTH} with GSC data:          {n_current:,}")
print(f"Pages with prior month data ({prior_month}): {n_joined:,}")
print(f"Pages dropped (no prior month):          {n_dropped:,} "
      f"({n_dropped/n_current*100:.1f}%)")
print()
print("These dropped pages are systematically newer content.")
print("The model cannot score them — this is a known limitation.")
print()

# Also show client count comparison
print("Client coverage:")
n_clients_current = current["client_hash_id"].nunique()
n_clients_joined = feat["client_hash_id"].nunique()
print(f"  Clients in {DEV_MONTH}:            {n_clients_current:,}")
print(f"  Clients with prior month data: {n_clients_joined:,}")
print(f"  Clients dropped:               {n_clients_current - n_clients_joined:,}")

DATA LIMIT — Unbalanced panel (missing prior month)
Pages in 2026-03 with GSC data:          176,738
Pages with prior month data (2026-02): 134,238
Pages dropped (no prior month):          42,500 (24.0%)

These dropped pages are systematically newer content.
The model cannot score them — this is a known limitation.

Client coverage:
  Clients in 2026-03:            47
  Clients with prior month data: 42
  Clients dropped:               5


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card

---

**Contract summary:**

- Unit of analysis: one content page per month (aggregated from daily rows)
- Table: `fact_content_daily_performance`, partitioned by month
- Development month: `2026-03` | Sealed test month: `2026-06` (never touched)
- Availability filter: `gsc_data_available IS TRUE`
- Proxy label: `is_declining` = impressions this month < impressions prior month
- Five features: monthly_impressions, monthly_clicks, avg_position, ctr,
  monthly_engaged_sessions
- Excluded: prior_impressions (leaky — used to construct the label),
  AI traffic columns (too sparse), sealed test month
- Named limitation: unbalanced panel — newer clients have no prior month
  data and are systematically excluded from the feature frame

**Leakage trap result:**
Adding `prior_impressions` as a feature inflated Precision@50.
That column is excluded from all future modeling.

**Next step:** w04 — EDA and signal audit on the warehouse data